In [3]:
# 1. Install required libraries
!pip install transformers accelerate scipy datasets librosa > /dev/null

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

import torch
import urllib.request
from transformers import pipeline, AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile

print("Loading Models... this takes about a minute.")

# 2. Load Whisper (Speech to Text)
transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if torch.cuda.is_available() else -1, chunk_length_s=30)

# 3. Load MusicGen (Text to Music)
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
music_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
if torch.cuda.is_available():
    music_model = music_model.to("cuda")

# 4. Define the 8-Axis Agent Logic (Simplified for Demo)
def determine_dna_and_music_prompt(transcription):
    text = transcription.lower()

    # Simple keyword-based DNA extraction
    if any(word in text for word in ["push", "kill", "attack", "fast", "rush"]):
        dna = "Aggressive/Rush"
        music_prompt = "High energy electronic battle music, heavy bass, fast tempo 140 bpm, aggressive synthesizer"
    elif any(word in text for word in ["hide", "wait", "quiet", "stealth", "sniper"]):
        dna = "Stealth/Tactical"
        music_prompt = "Dark ambient, tense strings, slow tempo, quiet atmospheric thriller music"
    else:
        dna = "Casual/Exploration"
        music_prompt = "Lo-fi chill gaming beats, relaxing, ambient, peaceful synthesizer"

    return dna, music_prompt

# 5. Execute Pipeline (Simulating player audio)
# (In a real scenario, you upload a .wav file. Here we use a sample from HuggingFace for speed)
print("\n--- Running Multimodal Pipeline ---")
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac" # Placeholder audio
urllib.request.urlretrieve(audio_url, "mlk.flac") # Download locally to avoid streaming issues
transcription = transcriber("mlk.flac")["text"]
print(f"1. Whisper Transcription: '{transcription}'")

# Override transcription for the sake of the gaming demo
simulated_voice_chat = "We need to push them now! Attack the base, go fast!"
print(f"1b. Simulated Gamer Chat: '{simulated_voice_chat}'")

dna, music_prompt = determine_dna_and_music_prompt(simulated_voice_chat)
print(f"2. Extracted Player DNA: {dna}")
print(f"3. Generated Music Prompt: {music_prompt}")

# 6. Generate Soundtrack
print("4. Generating Custom Soundtrack with MusicGen... (Takes ~10-20 seconds)")
inputs = processor(text=[music_prompt], padding=True, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

audio_values = music_model.generate(**inputs, max_new_tokens=256)

# 7. Save Output
sampling_rate = music_model.config.audio_encoder.sampling_rate
audio_data = audio_values[0, 0].cpu().numpy()
scipy.io.wavfile.write("custom_player_soundtrack.wav", rate=sampling_rate, data=audio_data)
print("5. DONE! Saved as 'custom_player_soundtrack.wav'.")

Loading Models... this takes about a minute.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Multimodal Pipeline ---


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

1. Whisper Transcription: ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'
1b. Simulated Gamer Chat: 'We need to push them now! Attack the base, go fast!'
2. Extracted Player DNA: Aggressive/Rush
3. Generated Music Prompt: High energy electronic battle music, heavy bass, fast tempo 140 bpm, aggressive synthesizer
4. Generating Custom Soundtrack with MusicGen... (Takes ~10-20 seconds)
5. DONE! Saved as 'custom_player_soundtrack.wav'.
